# ESG Report Text Analysis

This notebook compares the generated ESG reports for Amazon, Microsoft, SAP, and Zalando. The workflow uses classical Python text analysis inspired by the lecture notebook: regex preprocessing, stopword removal, `Counter` term frequencies, keyword lexicons, sentiment/tone lexicons, TF-IDF cosine similarity, and Jaccard similarity.


## Data basis

The input files are the four English ESG reports in `reports/generated`. The notebook writes tabular outputs to `results/*.csv` and visual outputs to `results/figures/*.svg`.


In [ ]:
from pathlib import Path
import csv, math, re
from collections import Counter
from docx import Document
import pandas as pd

PROJECT = Path(r'C:/Jann/Studium/2. Semester/ESG-Report')
REPORT_DIR = PROJECT / 'reports' / 'generated'
RESULTS_DIR = PROJECT / 'results'
REPORTS = {
    'Amazon': REPORT_DIR / 'Amazon_ESG_Report_Generated.docx',
    'Microsoft': REPORT_DIR / 'Microsoft_ESG_Report_Generated.docx',
    'SAP': REPORT_DIR / 'SAP_ESG_Report_Generated.docx',
    'Zalando': REPORT_DIR / 'Zalando_ESG_Report_Generated.docx',
}

def extract_docx_text(path):
    doc = Document(path)
    parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    for table in doc.tables:
        for row in table.rows:
            cells = [cell.text.strip() for cell in row.cells if cell.text.strip()]
            if cells:
                parts.append(' | '.join(cells))
    return '\n'.join(parts)

texts = {company: extract_docx_text(path) for company, path in REPORTS.items()}
{company: len(text) for company, text in texts.items()}


In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z]+(?:[-'][A-Za-z]+)?")
SENTENCE_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'])")
STOPWORDS = set('''a about above after again against all also am an and any are as at be because been before being below between both but by can could did do does doing down during each few for from further had has have having he her here hers him his how i if in into is it its itself just me more most my no nor not now of off on once only or other our ours out over own same she should so some such than that the their theirs them then there these they this those through to too under until up very was we were what when where which while who whom why will with within would you your'''.split())
DOMAIN_STOPWORDS = set('amazon microsoft sap zalando esg report reports generated company companies information general environmental environment social governance section task data source sources material analysis business group inc se ag com annual sustainability responsibility'.split())

def tokenize(text):
    return [m.group(0).lower().strip("-'") for m in TOKEN_RE.finditer(text)]

def preprocess(text):
    return [t for t in tokenize(text) if len(t) > 2 and t not in STOPWORDS and t not in DOMAIN_STOPWORDS]

tokens = {company: preprocess(text) for company, text in texts.items()}
top_terms = {company: Counter(tok).most_common(15) for company, tok in tokens.items()}
top_terms


In [ ]:
# Recompute classical text metrics, ESG keyword coverage, and tone tables from the raw DOCX texts.
ESG_LEXICONS = {'Environmental': ['climate', 'carbon', 'emission', 'emissions', 'scope 1', 'scope 2', 'scope 3', 'greenhouse gas', 'ghg', 'energy', 'renewable', 'electricity', 'water', 'waste', 'recycling', 'circular', 'circularity', 'packaging', 'biodiversity', 'net zero', 'decarbonization', 'decarbonisation', 'supplier decarbonization', 'climate pledge', 'data center', 'data centre', 'carbon free', 'carbon neutral', 'solar', 'wind', 'logistics', 'returns', 'materials', 'science based targets'], 'Social': ['employee', 'employees', 'workforce', 'workers', 'labour', 'labor', 'human rights', 'diversity', 'inclusion', 'equity', 'safety', 'health', 'wellbeing', 'well-being', 'training', 'reskilling', 'talent', 'employee engagement', 'communities', 'accessibility', 'supplier', 'suppliers', 'living wage', 'audit', 'audits', 'union', 'freedom of association', 'customer safety', 'content safety', 'responsible ai', 'digital inclusion', 'grievance', 'complaint', 'fair pay', 'harassment', 'discrimination'], 'Governance': ['board', 'supervisory board', 'committee', 'audit', 'compliance', 'ethics', 'code of conduct', 'anti-corruption', 'risk management', 'internal controls', 'data protection', 'cybersecurity', 'privacy', 'ai governance', 'dsa', 'csrd', 'double materiality', 'assurance', 'reporting', 'executive compensation', 'whistleblower', 'supply chain due diligence', 'regulatory', 'transparency', 'shareholder', 'agm', 'procurement controls']}
TONE_LEXICONS = {'Success': ['achieved', 'reduced', 'improved', 'progress', 'strong', 'robust', 'mature', 'leadership', 'exceeded', 'completed', 'expanded', 'increased', 'certified', 'recognized', 'integrated', 'strengthened', 'delivered', 'commitment', 'committed', 'enabled', 'implemented'], 'Risk': ['risk', 'risks', 'challenge', 'challenges', 'pressure', 'litigation', 'investigation', 'criticism', 'complaint', 'controversy', 'incident', 'breach', 'uncertainty', 'exposure', 'dependency', 'gap', 'constraint', 'concern', 'scrutiny', 'violation', 'fine', 'penalty'], 'Controversy': ['controversy', 'controversies', 'criticism', 'complaint', 'allegation', 'lawsuit', 'investigation', 'union', 'strike', 'dsa', 'antitrust', 'labor', 'privacy', 'incident', 'fine', 'protest'], 'Targets_Tradeoffs': ['target', 'targets', 'goal', 'goals', 'ambition', 'by 2030', 'by 2040', 'by 2050', 'roadmap', 'pathway', 'future', 'planned', 'expected', 'transition', 'trade off', 'trade-off', 'tension', 'conflict', 'requires', 'dependent']}
NUMBER_RE = re.compile(r"(?<![A-Za-z])(?:\d{1,3}(?:,\d{3})+|\d+(?:\.\d+)?)(?:\s?(?:%|percent|million|billion|bn|m|tco2e|co2e|eur|usd|\$|employees|suppliers|countries))?", re.IGNORECASE)

def normalize_phrase_text(text):
    return ' ' + re.sub(r'[^a-z0-9]+', ' ', text.lower()).strip() + ' '

def count_terms(text, terms):
    normalized = normalize_phrase_text(text)
    total = 0
    for term in terms:
        normalized_term = normalize_phrase_text(term).strip()
        total += len(re.findall(rf'\b{re.escape(normalized_term)}\b', normalized))
    return total

metrics_rows, esg_rows, tone_rows, top_term_rows = [], [], [], []
for company, text in texts.items():
    all_tokens = tokenize(text)
    clean_tokens = tokens[company]
    sentences = [s.strip() for s in SENTENCE_RE.split(text.replace('\n', ' ')) if s.strip()]
    number_count = len(NUMBER_RE.findall(text))
    metrics_rows.append({
        'company': company,
        'word_count': len(all_tokens),
        'sentence_count': len(sentences),
        'avg_sentence_length': round(len(all_tokens) / max(len(sentences), 1), 2),
        'unique_terms': len(set(clean_tokens)),
        'lexical_diversity': round(len(set(clean_tokens)) / max(len(clean_tokens), 1), 4),
        'numeric_mentions': number_count,
        'numeric_mentions_per_1000_words': round(number_count / max(len(all_tokens), 1) * 1000, 2),
    })
    esg_counts = {pillar: count_terms(text, terms) for pillar, terms in ESG_LEXICONS.items()}
    esg_rows.append({
        'company': company,
        'environmental_hits': esg_counts['Environmental'],
        'social_hits': esg_counts['Social'],
        'governance_hits': esg_counts['Governance'],
        'total_esg_hits': sum(esg_counts.values()),
        'esg_hits_per_1000_words': round(sum(esg_counts.values()) / max(len(all_tokens), 1) * 1000, 2),
        'strongest_pillar': max(esg_counts.items(), key=lambda item: item[1])[0],
    })
    tone_counts = {tone: count_terms(text, terms) for tone, terms in TONE_LEXICONS.items()}
    tone_rows.append({
        'company': company,
        'success_hits': tone_counts['Success'],
        'risk_hits': tone_counts['Risk'],
        'controversy_hits': tone_counts['Controversy'],
        'target_tradeoff_hits': tone_counts['Targets_Tradeoffs'],
        'risk_to_success_ratio': round(tone_counts['Risk'] / max(tone_counts['Success'], 1), 2),
    })
    for rank, (term, count) in enumerate(Counter(clean_tokens).most_common(20), start=1):
        top_term_rows.append({'company': company, 'rank': rank, 'term': term, 'count': count})

metrics = pd.DataFrame(metrics_rows)
esg_coverage = pd.DataFrame(esg_rows)
tone = pd.DataFrame(tone_rows)
top_terms_df = pd.DataFrame(top_term_rows)
metrics.to_csv(RESULTS_DIR / 'text_metrics.csv', index=False)
esg_coverage.to_csv(RESULTS_DIR / 'esg_keyword_coverage.csv', index=False)
tone.to_csv(RESULTS_DIR / 'tone_metrics.csv', index=False)
top_terms_df.to_csv(RESULTS_DIR / 'top_terms.csv', index=False)
display(metrics)
display(esg_coverage)
display(tone)


## Visual comparison

![Word count](results/figures/word_counts.svg)

![Average sentence length](results/figures/average_sentence_length.svg)

![ESG keyword coverage](results/figures/esg_keyword_coverage.svg)

![Tone profile](results/figures/tone_profile.svg)

![Top terms](results/figures/top_terms.svg)

![Keyword cloud](results/figures/keyword_cloud.svg)


In [ ]:
# Recompute TF-IDF cosine similarity and Jaccard similarity without scikit-learn.
def tfidf_vectors(token_docs):
    labels = list(token_docs)
    vocab = sorted(set(token for doc_tokens in token_docs.values() for token in doc_tokens))
    doc_count = len(labels)
    df = {term: sum(1 for doc_tokens in token_docs.values() if term in set(doc_tokens)) for term in vocab}
    vectors = {}
    for label, doc_tokens in token_docs.items():
        counts = Counter(doc_tokens)
        total = max(len(doc_tokens), 1)
        vectors[label] = {term: (counts[term] / total) * (math.log((1 + doc_count) / (1 + df[term])) + 1) for term in vocab}
    return vectors

def cosine_similarity(vec_a, vec_b):
    terms = set(vec_a) | set(vec_b)
    dot = sum(vec_a.get(term, 0) * vec_b.get(term, 0) for term in terms)
    norm_a = math.sqrt(sum(value * value for value in vec_a.values()))
    norm_b = math.sqrt(sum(value * value for value in vec_b.values()))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0

vectors = tfidf_vectors(tokens)
labels = list(tokens)
token_sets = {label: set(doc_tokens) for label, doc_tokens in tokens.items()}
cosine_rows, jaccard_rows = [], []
for left in labels:
    cosine_row = {'company': left}
    jaccard_row = {'company': left}
    for right in labels:
        cosine_row[right] = round(cosine_similarity(vectors[left], vectors[right]), 4)
        union = token_sets[left] | token_sets[right]
        jaccard_row[right] = round(len(token_sets[left] & token_sets[right]) / len(union), 4) if union else 0
    cosine_rows.append(cosine_row)
    jaccard_rows.append(jaccard_row)
cosine = pd.DataFrame(cosine_rows)
jaccard = pd.DataFrame(jaccard_rows)
cosine.to_csv(RESULTS_DIR / 'tfidf_cosine_similarity.csv', index=False)
jaccard.to_csv(RESULTS_DIR / 'jaccard_similarity.csv', index=False)
display(cosine)
display(jaccard)


## Similarity matrices

TF-IDF cosine similarity compares weighted term profiles, while Jaccard similarity compares set overlap after preprocessing. Together they show whether reports are close because they use the same vocabulary or because they share the same distinctive terms.

![TF-IDF cosine similarity](results/figures/tfidf_cosine_similarity.svg)

![Jaccard similarity](results/figures/jaccard_similarity.svg)


## Interpretation notes

- Word and sentence metrics indicate report length and density, not quality by themselves.
- ESG keyword coverage is a transparent proxy for thematic balance across E, S, and G.
- Tone lexicons approximate whether the report emphasizes achievements, risks, controversies, or future targets and tradeoffs.
- The qualitative Word document in `results/ESG_LLM_Qualitative_Analysis.docx` interprets these metrics together with a close reading of the reports.
